# Target Problem

The project proposes to use a deep reinforcement learning framework to learn a profitable stock trading strategy, with the goal to optimize the cumulative return and Alpha. It would select S&P500 Index with the top 20 market capitalization stocks as our trading stock pool. The input to the algorithm is the market trend for these stocks in the last month, remaining balance, and current portfolio. The model agent output is a series of trading actions among stocks. The available trading
action options are: sell, buy and hold. The market data in the most recent months will be used to feed
the model performance evaluation.

The algorithm is trained using Deep Reinforcement Learning (DRL) algorithms and the components of the reinforcement learning environment are:

* Action: The action space describes the allowed actions that the agent interacts with the
environment. Normally, a ∈ A includes three actions: a ∈ {−1, 0, 1}, where −1, 0, 1 represent
selling, holding, and buying one stock. Also, an action can be carried upon multiple shares. We use
an action space {−k, ..., −1, 0, 1, ..., k}, where k denotes the number of shares. For example, "Buy
10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or −10, respectively

* Reward function: r(s, a, s′) is the incentive mechanism for an agent to learn a better action. The change of the portfolio value when action a is taken at state s and arriving at new state s',  i.e., r(s, a, s′) = v′ − v, where v′ and v represent the portfolio
values at state s′ and s, respectively

* State: The state space describes the observations that the agent receives from the environment. Just as a human trader needs to analyze various information before executing a trade, so
our trading agent observes many different features to better learn in an interactive environment.

* Environment: SP500 top 20 companies


The data of the single stock that we will be using for this case study is obtained from Yahoo Finance API. The data contains Open-High-Low-Close price and volume.


## Environment Setup

### Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta

%matplotlib inline
from rl.config import config
# from rl.marketdata.yahoodownloader import YahooDownloader
# from finrl.meta.data_processors import YahooDownloader
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.meta.preprocessor.preprocessors import data_split
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv

from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

# from rl.preprocessing.preprocessors import FeatureEngineer
# from rl.preprocessing.data import data_split
# from rl.env.env_stocktrading import StockTradingEnv
# from rl.model.models import DRLAgent
# from rl.trade.backtest import backtest_stats, backtest_plot, get_daily_return, get_baseline

from pprint import pprint

import itertools

/Users/kennethwang/MyFiles/code/RL-stock-trading/.venv/lib/python3.10/site-packages/gym/envs/registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/kennethwang/MyFiles/code/RL-stock-trading/.venv/lib/python3.10/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.ass

### Cache Folders

In [2]:
import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [11]:
YahooDownloader(start_date = "2012-01-01",
                     end_date = "2019-01-01",
                     ticker_list = ["GOOGL"]).fetch_data()

[*********************100%***********************]  1 of 1 completed

1 Failed download:
- GOOGL: No data found for this date range, symbol may be delisted


ValueError: no data is fetched.

### Fetch Data

In [6]:
# from config.py start_date is a string
start_date = "2012-01-01" # config.START_DATE
# from config.py end_date is a string
end_date = "2019-01-01" # config.END_DATE
# from config.py split_date is a string
split_date = "2015-01-01" # config.SPLIT_DATE
# config target ticker
target_ticker = config.SP500_20_TICKER
# config tech_indicator_list
tech_indicator_list = config.TECHNICAL_INDICATORS_LIST
print("training period: {}-{}, testing period: {}-{}\n".format(start_date, split_date, split_date, end_date))
print("target ticker list:\n {}\n".format(target_ticker))
print("tech_indicator_list:\n {}\n".format(tech_indicator_list))
df = YahooDownloader(start_date = start_date,
                     end_date = end_date,
                     ticker_list = target_ticker).fetch_data()
df.sort_values(['date','tic'],ignore_index=True).head()

training period: 2012-01-01-2015-01-01, testing period: 2015-01-01-2019-01-01

target ticker list:
 ['AAPL', 'MSFT', 'AMZN', 'BRK-B', 'JPM', 'JNJ', 'UNH', 'HD', 'PG', 'NVDA', 'DIS', 'BAC', 'CMCSA', 'XOM', 'VZ', 'T', 'ADBE', 'PFE', 'CSCO', 'INTC']

tech_indicator_list:
 ['macd', 'boll_ub', 'boll_lb', 'rsi_10', 'rsi_20', 'cci_10', 'cci_20', 'dx_30', 'close_20_sma', 'close_60_sma', 'close_120_sma', 'close_20_ema', 'close_60_ema', 'close_120_ema']

[*********************100%***********************]  1 of 1 completed

1 Failed download:
- AAPL: No data found for this date range, symbol may be delisted
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- MSFT: No data found for this date range, symbol may be delisted
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- AMZN: No data found for this date range, symbol may be delisted
[*********************100%***********************]  1 of 1 completed

1 Failed downloa

ValueError: no data is fetched.

## Data Preprocessing
Data preprocessing is a crucial step for training a high quality machine learning model. We need to check for missing data and do feature engineering in order to convert the data into a model-ready state.
* Add technical indicators. In practical trading, various information needs to be taken into account, for example the historical stock prices, current holding shares, technical indicators, etc. In this article, we demonstrate two trend-following technical indicators: MACD and RSI.
* Add turbulence index. Risk-aversion reflects whether an investor will choose to preserve the capital. It also influences one's trading strategy when facing different market volatility level. To control the risk in a worst-case scenario, such as financial crisis of 2007–2008, FinRL employs the financial turbulence index that measures extreme asset price fluctuation.

### Feature Engineering

In [8]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = tech_indicator_list,
                    use_turbulence=True,
                    user_defined_feature = False)

processed = fe.preprocess_data(df)

NameError: name 'df' is not defined

In [ ]:
print(processed.shape)
processed.head()

(105680, 23)


,date,close,high,low,open,volume,tic,day,macd,boll_ub,...,cci_10,cci_20,dx_30,close_20_sma,close_60_sma,close_120_sma,close_20_ema,close_60_ema,close_120_ema,turbulence
0,2000-01-03,0.841048,0.845274,0.764034,0.787983,535796800,AAPL,0,0.0,0.905873,...,-66.666667,-66.666667,100.0,0.841048,0.841048,0.841048,0.841048,0.841048,0.841048,0.0
1,2000-01-03,16.274668,16.755616,15.948864,16.693558,7384400,ADBE,0,0.0,0.905873,...,-66.666667,-66.666667,100.0,16.274668,16.274668,16.274668,16.274668,16.274668,16.274668,0.0
2,2000-01-03,4.468750,4.478125,3.952344,4.075000,322352000,AMZN,0,0.0,0.905873,...,-66.666667,-66.666667,100.0,4.468750,4.468750,4.468750,4.468750,4.468750,4.468750,0.0
3,2000-01-03,12.560280,13.030277,12.446832,13.030277,13705800,BAC,0,0.0,0.905873,...,-66.666667,-66.666667,100.0,12.560280,12.560280,12.560280,12.560280,12.560280,12.560280,0.0
4,2000-01-03,35.299999,36.580002,34.820000,36.500000,875000,BRK-B,0,0.0,0.905873,...,-66.666667,-66.666667,100.0,35.299999,35.299999,35.299999,35.299999,35.299999,35.299999,0.0


: 

: 

: 

: 

: 

: 

: 

### Data display

In [5]:
list_ticker = processed["tic"].unique().tolist()
print(list_ticker)
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))
processed_full = pd.DataFrame(combination,columns=["date","tic"]).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])
processed_full = processed_full.fillna(0)

NameError: name 'processed' is not defined

In [ ]:
processed_full.sort_values(['date','tic'],ignore_index=True).head(10)

,date,tic,close,high,low,open,volume,day,macd,boll_ub,...,cci_10,cci_20,dx_30,close_20_sma,close_60_sma,close_120_sma,close_20_ema,close_60_ema,close_120_ema,turbulence
0,2000-01-03,AAPL,0.841048,0.845274,0.764034,0.787983,535796800.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,0.841048,0.841048,0.841048,0.841048,0.841048,0.841048,0.0
1,2000-01-03,ADBE,16.274668,16.755616,15.948864,16.693558,7384400.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,16.274668,16.274668,16.274668,16.274668,16.274668,16.274668,0.0
2,2000-01-03,AMZN,4.468750,4.478125,3.952344,4.075000,322352000.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,4.468750,4.468750,4.468750,4.468750,4.468750,4.468750,0.0
3,2000-01-03,BAC,12.560280,13.030277,12.446832,13.030277,13705800.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,12.560280,12.560280,12.560280,12.560280,12.560280,12.560280,0.0
4,2000-01-03,BRK-B,35.299999,36.580002,34.820000,36.500000,875000.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,35.299999,35.299999,35.299999,35.299999,35.299999,35.299999,0.0
5,2000-01-03,CMCSA,10.811822,11.332176,10.450464,11.202088,2333700.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,10.811822,10.811822,10.811822,10.811822,10.811822,10.811822,0.0
6,2000-01-03,CSCO,35.360298,36.076094,33.887805,35.973837,53076000.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,35.360298,35.360298,35.360298,35.360298,35.360298,35.360298,0.0
7,2000-01-03,DIS,22.736227,22.783793,21.880052,22.260574,8402230.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,22.736227,22.736227,22.736227,22.736227,22.736227,22.736227,0.0
8,2000-01-03,HD,38.380375,40.735451,37.570818,40.404268,12030800.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,38.380375,38.380375,38.380375,38.380375,38.380375,38.380375,0.0
9,2000-01-03,INTC,24.710649,24.817161,23.645535,23.649973,57710200.0,0.0,0.0,0.905873,...,-66.666667,-66.666667,100.0,24.710649,24.710649,24.710649,24.710649,24.710649,24.710649,0.0


: 

: 

: 

: 

: 

: 

: 

### Data Split

In [ ]:
train = data_split(processed_full, start_date, split_date)
evaluate = data_split(processed_full, split_date, end_date)
print(train.shape)
print(evaluate.shape)

(95580, 23)
(10100, 23)


: 

: 

: 

: 

: 

: 

Rolling Predict Time Periods

In [ ]:
predict_start_date=['2019-01-01', '2020-01-01']
predict_end_date=['2020-01-01', '2021-01-01']

: 

: 

: 

: 

: 

: 

## RL Environment
Considering the stochastic and interactive nature of the automated stock trading tasks, a financial task is modeled as a **Markov Decision Process (MDP)** problem. The training process involves observing stock price change, taking an action and reward's calculation to have the agent adjusting its strategy accordingly. By interacting with the environment, the trading agent will derive a trading strategy with the maximized rewards as time proceeds.

Our trading environments, based on OpenAI Gym framework, simulate live stock markets with real market data according to the principle of time-driven simulation.

The action space describes the allowed actions that the agent interacts with the environment. Normally, action a includes three actions: {-1, 0, 1}, where -1, 0, 1 represent selling, holding, and buying one share. Also, an action can be carried upon multiple shares. We use an action space {-k,…,-1, 0, 1, …, k}, where k denotes the number of shares to buy and -k denotes the number of shares to sell. For example, "Buy 10 shares of AAPL" or "Sell 10 shares of AAPL" are 10 or -10, respectively. The continuous action space needs to be normalized to [-1, 1], since the policy is defined on a Gaussian distribution, which needs to be normalized and symmetric.

In [ ]:
stock_dimension = len(processed_full.tic.unique())
state_space = 1 + 2*stock_dimension + len(tech_indicator_list)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 20, State Space: 321


: 

: 

: 

: 

: 

: 

### Basic Model Train

In [ ]:
def train_predict(model_name, tb_log_name, dataset, start, split, end, env_params, model_kwargs, total_timesteps):
    e_train_gym = StockTradingEnv(df = data_split(dataset, start, split), **env_params)
    e_evaluation_gym = StockTradingEnv(df=data_split(dataset, split, end), **env_params)
    
    env_train, _ = e_train_gym.get_sb_env()
    
    agent = DRLAgent(env = env_train)
    model = agent.get_model(model_name, model_kwargs = model_kwargs)
    
    trained_model = agent.train_model(
        model=model, 
        tb_log_name=tb_log_name,
        total_timesteps=total_timesteps)

    train_value, train_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    test_value, test_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    return train_value, train_actions, test_value, test_actions

: 

: 

: 

: 

: 

: 

In [ ]:
def rolling_predict(model_name, tb_log_name, dataset, start, split, end, env_params, model_kwargs, total_timesteps):
    e_train_gym = StockTradingEnv(df = data_split(dataset, start, split), **env_params)
    e_evaluation_gym = StockTradingEnv(df=data_split(dataset, split, end), **env_params)
    
    env_train, _ = e_train_gym.get_sb_env()
    
    agent = DRLAgent(env = env_train)
    model = agent.get_model(model_name, model_kwargs = model_kwargs)
    
    trained_model = agent.train_model(
        model=model, 
        tb_log_name=tb_log_name,
        total_timesteps=total_timesteps)

    test_value, test_actions = DRLAgent.DRL_prediction(trained_model, e_evaluation_gym)
    return test_value, test_actions

: 

: 

: 

: 

: 

: 

In [ ]:
def train_and_predict(model_name, dataset, env_params, model_kwargs, total_timesteps):
    start = start_date
    split=predict_start_date[0]
    end=predict_end_date[0]
    
    
    print("rolling predict: {}:{}:{}, env_params: {}".format(start, split, end, env_params))
    train_value, train_actions, test_value, test_actions = train_predict(model_name, "{}_train_{}".format(model_name, 0), dataset, start, split, end, env_params, model_kwargs, total_timesteps)
    
    rolling_env_params = env_params.copy()
    
    for i in range(1, len(predict_start_date)):
        rolling_env_params['initial_amount']=test_value.iloc[-1].account_value
        
        start = start_date
        split=predict_start_date[i]
        end=predict_end_date[i]
        
        print("rolling predict: {}:{}:{}, env_params: {}".format(start, split, end, rolling_env_params))
        
        value, actions = rolling_predict(model_name, "{}_rolling_{}".format(model_name, i), dataset, start, split, end, rolling_env_params, model_kwargs, total_timesteps)
        test_value=pd.concat([test_value, value])
        test_actions=pd.concat([test_actions, actions])
    return train_value, train_actions, test_value, test_actions

: 

: 

: 

: 

: 

: 

In [ ]:
ENV_PARAMS = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": tech_indicator_list, 
    "action_space": stock_dimension, 
    "reward_scaling": 1e-6
}
A2C_PARAMS = {
    "n_steps": 5, 
    "ent_coef": 0.005, 
    "learning_rate": 0.0001
}
PPO_PARAMS = {
    "n_steps": 2048,
    "ent_coef": 0.005,
    "learning_rate": 0.0001,
    "batch_size": 128,
}
DDPG_PARAMS = {
    "batch_size": 128, 
    "buffer_size": 100000, 
    "learning_rate": 0.0005
}
TD3_PARAMS = {
    "batch_size": 128, 
    "buffer_size": 100000, 
    "learning_rate": 0.0005
}
SAC_PARAMS = {
    "batch_size": 128,
    "buffer_size": 100000,
    "learning_rate": 0.0001,
    "learning_starts": 100,
    "ent_coef": "auto_0.1"
}
total_timesteps = 50000

: 

: 

: 

: 

: 

: 

In [ ]:
df_value, df_actions = dict(), dict()

: 

: 

: 

: 

: 

: 

In [ ]:
%%time
%%capture

df_value['train_a2c'],df_actions['train_a2c'],df_value['test_a2c'],df_actions['test_a2c']=train_and_predict("a2c", processed_full, ENV_PARAMS, A2C_PARAMS, total_timesteps=100000)

TypeError: StockTradingEnv.__init__() got an unexpected keyword argument 'df'

CPU times: user 35.3 ms, sys: 5.99 ms, total: 41.3 ms
Wall time: 40.9 ms


: 

: 

: 

: 

: 

: 

In [ ]:
%%time
%%capture

df_value['train_ppo'],df_actions['train_ppo'],df_value['test_ppo'],df_actions['test_ppo']=train_and_predict("ppo", processed_full, ENV_PARAMS, PPO_PARAMS, total_timesteps=100000)

TypeError: StockTradingEnv.__init__() got an unexpected keyword argument 'df'

CPU times: user 29.9 ms, sys: 5.54 ms, total: 35.4 ms
Wall time: 34.5 ms


: 

: 

: 

: 

: 

: 

In [ ]:
%%time
%%capture

df_value['train_ddpg'],df_actions['train_ddpg'],df_value['test_ddpg'],df_actions['test_ddpg']=train_and_predict("ddpg", processed_full, ENV_PARAMS, DDPG_PARAMS, total_timesteps=100000)

TypeError: StockTradingEnv.__init__() got an unexpected keyword argument 'df'

CPU times: user 29.9 ms, sys: 6.17 ms, total: 36 ms
Wall time: 35.1 ms


: 

: 

: 

: 

: 

: 

In [ ]:
%%time
%%capture

df_value['train_td3'],df_actions['train_td3'],df_value['test_td3'],df_actions['test_td3']=train_and_predict("td3", processed_full, ENV_PARAMS, TD3_PARAMS, total_timesteps=100000)

TypeError: StockTradingEnv.__init__() got an unexpected keyword argument 'df'

CPU times: user 41.3 ms, sys: 11.8 ms, total: 53.1 ms
Wall time: 63.7 ms


: 

: 

: 

: 

: 

: 

In [ ]:
%%time
%%capture

df_value['train_sac'],df_actions['train_sac'],df_value['test_sac'],df_actions['test_sac']=train_and_predict("sac", processed_full, ENV_PARAMS, SAC_PARAMS, total_timesteps=100000)

TypeError: StockTradingEnv.__init__() got an unexpected keyword argument 'df'

CPU times: user 36.2 ms, sys: 5.84 ms, total: 42 ms
Wall time: 41.4 ms


: 

: 

: 

: 

: 

: 

## Evaluation
Backtesting plays a key role in evaluating the performance of a trading strategy. Automated backtesting tool is preferred because it reduces the human error. We usually use the Quantopian pyfolio package to backtest our trading strategies. It is easy to use and consists of various individual plots that provide a comprehensive image of the performance of a trading strategy.

### Baseline Performance

In [ ]:
# get baseline stats
print("baseline performance backtest:")
training_baseline_df = get_baseline(ticker="SPY", start=start_date,end=split_date)
evaluation_baseline_df = get_baseline(ticker="SPY", start=split_date,end=end_date)

print()
print("================Training Period Perf================")
stats = backtest_stats(training_baseline_df, value_col_name='close')
print()
print("================Evaluation Period Perf================")
stats = backtest_stats(evaluation_baseline_df, value_col_name='close')

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

baseline performance backtest:
Shape of DataFrame:  (4779, 8)
Shape of DataFrame:  (505, 8)

================Training Period Perf================
Annual return          0.048459
Cumulative returns     1.453271
Annual volatility      0.192484
Sharpe ratio           0.342078
Calmar ratio           0.087805
Stability              0.743452
Max drawdown          -0.551894
Omega ratio            1.067161
Sortino ratio          0.484343
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.899599
Daily value at risk   -0.023989
dtype: float64

================Evaluation Period Perf================
Annual return          0.244922
Cumulative returns     0.551179
Annual volatility      0.252380
Sharpe ratio           0.997195
Calmar ratio           0.726400
Stability              0.604514
Max drawdown          -0.337173
Omega ratio            1.234938
Sortino ratio          1.372113
Skew                        NaN
Kurtosis                    NaN
Tail ratio     

: 

: 

: 

: 

: 

: 

### Evaluation Performance

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_a2c'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_a2c'

: 

: 

: 

: 

: 

: 

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_ppo'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_ppo'

: 

: 

: 

: 

: 

: 

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_ddpg'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_ddpg'

: 

: 

: 

: 

: 

: 

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_td3'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_td3'

: 

: 

: 

: 

: 

: 

In [ ]:
%matplotlib inline
backtest_plot(df_value['test_sac'], baseline_ticker = 'SPY', baseline_start = split_date, baseline_end = end_date)

KeyError: 'test_sac'

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 

: 